# Notebook 03 of 7 — Basket X-Ray + Risk

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

In NB02 I built a per-name opinion of MSFT. Now I need to answer the question NB02 kept dodging: what do all 10 of my positions look like *together*? Concentration, risk, drawdown — the numbers I've been ignoring because I only have 10 tickers so 'obviously' I'm diversified.

By the end of this notebook we will be able to answer one question:

> *What am I actually exposed to at the basket level — after looking through my ETFs?*


In [1]:
# [Phase B / NB03 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio. See NB01 §0 for setup."
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 1. Load the basket

The 10-position basket from NB01. If `.notebook_state/basket.json`
doesn't exist yet (running NB03 standalone), we regenerate it from the
same locked list.

> **📖 Portfolio** — the full set of investments you hold, viewed as one thing rather than a list of tickers. Concentration, risk, and drawdown are portfolio-level questions; per-name P&L is not. [Investopedia →](https://www.investopedia.com/terms/p/portfolio.asp)
>
> **📖 Position** — a specific holding: ticker + quantity + entry basis. Ten positions is what my brokerage screen shows; whether it's actually ten *independent* bets is the whole point of NB03. [Investopedia →](https://www.investopedia.com/terms/p/position.asp)
>
> **📖 Weight** — a position's share of total portfolio value. Weights, not share counts, drive every risk number that follows. A 400-share position in a $30 name is smaller than 20 shares of a $900 name. [Investopedia →](https://www.investopedia.com/terms/w/weighted.asp)

*The code cell below loads the basket and prints a positions table.*

In [2]:
# [Phase B / NB03 §1] Load the 10-position basket from NB01
# Falls back to the STORY_BIBLE locked list if NB01 hasn't been run
# in this checkout — every notebook must be runnable standalone.
import json
from pathlib import Path

state = Path(".notebook_state")
basket_path = state / "basket.json"

BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12, "kind": "equity"},
    {"symbol": "NVDA",  "weight": 0.10, "kind": "equity"},
    {"symbol": "GOOGL", "weight": 0.08, "kind": "equity"},
    {"symbol": "AAPL",  "weight": 0.08, "kind": "equity"},
    {"symbol": "AMD",   "weight": 0.06, "kind": "equity"},
    {"symbol": "QQQ",   "weight": 0.15, "kind": "etf"},
    {"symbol": "VTI",   "weight": 0.20, "kind": "etf"},
    {"symbol": "VNQ",   "weight": 0.08, "kind": "etf"},
    {"symbol": "BND",   "weight": 0.10, "kind": "etf"},
    {"symbol": "GLD",   "weight": 0.03, "kind": "etf"},
]

if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    # NB01 uses "ticker" key; portfolio_intel uses "symbol". Normalize.
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {basket_path}")
else:
    basket = BASKET_LOCKED
    state.mkdir(exist_ok=True)
    basket_path.write_text(json.dumps(basket, indent=2), encoding="utf-8")
    print(f"Regenerated basket from STORY_BIBLE locked list -> {basket_path}")

total_w = sum(p["weight"] for p in basket)
print(f"Positions: {len(basket)}    Total weight: {total_w*100:.1f}%")
print()
print(f"{'Symbol':<8}{'Weight':>8}   Kind")
print("-" * 40)
for p in basket:
    kind = p.get("kind", "?")
    print(f"{p['symbol']:<8}{p['weight']*100:>7.1f}%   {kind}")


Loaded basket from .notebook_state\basket.json
Positions: 10    Total weight: 100.0%

Symbol    Weight   Kind
----------------------------------------
MSFT       12.0%   equity
NVDA       10.0%   equity
GOOGL       8.0%   equity
AAPL        8.0%   equity
AMD         6.0%   equity
QQQ        15.0%   etf
VTI        20.0%   etf
VNQ         8.0%   etf
BND        10.0%   etf
GLD         3.0%   etf


## 2. The naive view — sector pie WITHOUT look-through

Before we look through the ETFs, what does the raw sector view say?
This is what a spreadsheet would tell me: MSFT is Tech, NVDA is Tech,
AMD is Tech, QQQ is a "Tech ETF" (so bucket it Tech), VTI is a "Broad
ETF" (its own bucket), VNQ is "REIT", BND is "Bond Fund", GLD is
"Commodity". A rough count says maybe ~50% Tech once we include the
"Tech ETF" bucket.

Hold that number. The story I've heard is that ETFs hide
concentration — and that when we do the real look-through the Tech
exposure goes up because QQQ+VTI are secretly all NVDA/MSFT/AAPL
overlap. Let's see if that's actually true for MY basket.

> **📖 Diversification** — spreading capital so that no single name, sector, or macro shock can hurt you disproportionately. The textbook version says "own uncorrelated things"; the practical version is *the numbers in §5 and §7 below.* Just owning "a lot of tickers" is not diversification if they all move together. [Investopedia →](https://www.investopedia.com/terms/d/diversification.asp)
>
> **📖 Sector breakdown** — the percentage of the portfolio in each GICS sector. A rule of thumb: no single sector above ~30% unless you're deliberately taking that bet. My naive ~50% Tech is already a yellow flag before we even look through the ETFs. [Investopedia →](https://www.investopedia.com/terms/s/sectorbreakdown.asp)

*The code cell below classifies each position by its own sector and
prints the raw Tech percentage.*

In [3]:
# [Phase B / NB03 §2] Naive sector view — each ETF is its own "row"
# This is what a spreadsheet would tell me: MSFT is Tech, QQQ is
# labeled "Tech ETF", VNQ is "REIT", etc. No look-through.
# Sector labels come from fmp_cached's EquityInfo/EtfInfo for equities;
# for ETFs I hand-label since sector-of-ETF isn't a clean concept
# without look-through.
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

NAIVE_SECTOR_LABELS = {
    # Equities — via fmp_cached
    # ETFs — hand-labeled with their "obvious" bucket
    "QQQ": "Tech ETF",
    "VTI": "Broad ETF",
    "VNQ": "REIT",
    "BND": "Bond Fund",
    "GLD": "Commodity",
}

naive_by_sector: dict[str, float] = {}
for pos in basket:
    sym = pos["symbol"]
    w = pos["weight"]
    if sym in NAIVE_SECTOR_LABELS:
        sector = NAIVE_SECTOR_LABELS[sym]
    else:
        try:
            info = obb.equity.profile(symbol=sym, provider="fmp_cached").to_df()
            sector = info["sector"].iloc[0] if "sector" in info.columns else "Unknown"
        except Exception:
            sector = "Unknown"
    naive_by_sector[sector] = naive_by_sector.get(sector, 0.0) + w

print("Naive sector view (each ETF is its own bucket):")
print(f"{'Sector':<20}{'Weight':>10}")
print("-" * 32)
for sector, w in sorted(naive_by_sector.items(), key=lambda kv: -kv[1]):
    print(f"{sector:<20}{w*100:>9.1f}%")
tech_naive = naive_by_sector.get("Technology", 0.0) + naive_by_sector.get("Tech ETF", 0.0)
print()
print(f"Raw Tech (equity 'Technology' + 'Tech ETF' bucket): {tech_naive*100:.1f}%")
print("Hold that number.")


Naive sector view (each ETF is its own bucket):
Sector                  Weight
--------------------------------
Technology               36.0%
Broad ETF                20.0%
Tech ETF                 15.0%
Bond Fund                10.0%
Communication Services      8.0%
REIT                      8.0%
Commodity                 3.0%

Raw Tech (equity 'Technology' + 'Tech ETF' bucket): 51.0%
Hold that number.


## 3. The x-ray — look-through via `obb.portfolio_intel.xray.look_through`

Now the real view. `look_through` reads each ETF's holdings via
`EtfHoldings`, re-weights each underlying position by (basket weight ×
ETF weight-in-basket), and returns a *flattened* portfolio: what the
basket actually owns, not what it looks like on the tab labels.

QQQ's top holdings are MSFT, NVDA, AAPL, GOOGL, AMD — the exact names
already in my basket at full weight. VTI has a ~30% tech tilt in 2026
and MSFT is one of its top three positions. The moment the flattened
view lands, my "10 things" is closer to 47 things — but weighted so
that a handful of names dominate.

> **📖 Look-through** — the accounting/analysis discipline of expanding a pooled holding (ETF, mutual fund, holdco) into its underlying constituents before measuring exposure. Without it, "I own QQQ" and "I own MSFT" look like independent bets when they aren't. [Investopedia →](https://www.investopedia.com/terms/l/look-through-earnings.asp)

*The code cell below runs the look-through and prints the top-15
effective positions after flattening.*

In [4]:
# [Phase B / NB03 §3] Look-through via snapshot-backed ETF holdings
# NOTE on provider tier: obb.portfolio_intel.xray.look_through would
# call obb.etf.holdings(provider="fmp_cached") for each ETF. Without
# the ETF-Holdings sub-plan, that exhausts all FMP tiers per ETF and
# stalls the notebook. We do the flatten in-notebook using the
# snapshot-backed YFinanceEtfHoldingsRecordedFetcher + BondLadderFetcher
# (the exact fallback path NB01 §4 introduced).
from openbb_yfinance.models.recorded_etf_holdings import (
    YFinanceEtfHoldingsRecordedFetcher,
)
from openbb_yfinance.models.bond_ladder import YFinanceBondLadderFetcher

# Which basket rows are equity ETFs vs bond ETFs vs commodities vs equities
EQUITY_ETFS = {"QQQ", "VTI", "VNQ", "SPY", "DIA", "IWM", "VOO", "VEA"}
BOND_ETFS = {"BND", "AGG"}
COMMODITY_TRUSTS = {"GLD", "SLV"}

def _unwrap_position(pos: dict) -> list[tuple[str, float]]:
    """Return list of (underlying_symbol, effective_weight) rows."""
    sym = pos["symbol"]
    w = pos["weight"]
    if sym in EQUITY_ETFS:
        try:
            rows = YFinanceEtfHoldingsRecordedFetcher.fetch_from_snapshot(sym)
            # Only include rows with a real ticker + positive weight
            return [
                (r.symbol, w * r.weight)
                for r in rows
                if r.symbol and r.weight and r.weight > 0
            ]
        except Exception as exc:
            print(f"  WARN: {sym} equity-etf snapshot missing ({exc!s:.60}); "
                  f"leaving as opaque")
            return [(sym, w)]
    if sym in BOND_ETFS:
        try:
            bl = YFinanceBondLadderFetcher.fetch_from_snapshot(sym)
            # Bond holdings don't have equity tickers — treat as one
            # bond-basket row keyed by ETF symbol (this is honest;
            # the reader shouldn't expect equity-style unwrap for a bond ETF).
            return [(f"BOND_{sym}", w)]
        except Exception as exc:
            print(f"  WARN: {sym} bond-etf snapshot missing ({exc!s:.60})")
            return [(sym, w)]
    if sym in COMMODITY_TRUSTS:
        # Physical commodity — no look-through possible or meaningful
        return [(sym, w)]
    # Single equity — passes through as-is
    return [(sym, w)]

# Flatten every basket row into effective (symbol, weight) tuples
effective_rows: list[tuple[str, float]] = []
for pos in basket:
    effective_rows.extend(_unwrap_position(pos))

# Aggregate by underlying symbol (a name held both directly + via ETF
# gets its weights summed — that IS the concentration hiding in a
# naive count of "10 things")
effective: dict[str, float] = {}
for sym, w in effective_rows:
    effective[sym] = effective.get(sym, 0.0) + w

total_effective = sum(effective.values())
print(f"Basket: {len(basket)} tickers.  "
      f"After look-through: {len(effective)} distinct effective positions.  "
      f"Total effective weight: {total_effective*100:.1f}%")
print()
print("Top 15 effective positions (post-look-through):")
print(f"{'Symbol':<14}{'Effective weight':>18}")
print("-" * 34)
for sym, w in sorted(effective.items(), key=lambda kv: -kv[1])[:15]:
    print(f"{sym:<14}{w*100:>17.2f}%")


Basket: 10 tickers.  After look-through: 23 distinct effective positions.  Total effective weight: 72.6%

Top 15 effective positions (post-look-through):
Symbol          Effective weight
----------------------------------
MSFT                      14.01%
NVDA                      12.43%
AAPL                      10.50%
BOND_BND                  10.00%
GOOGL                      9.00%
AMD                        6.00%
GLD                        3.00%
AMZN                       1.41%
VRTPX                      1.15%
WELL                       0.67%
PLD                        0.53%
AVGO                       0.49%
GOOG                       0.45%
EQIX                       0.43%
MU                         0.36%


## 4. Sector pie WITH look-through — the pivot

Same chart as §2, but on the flattened portfolio.

Here's the surprise. The story I'd been told predicted Tech going
**up** post-look-through — because QQQ + VTI are supposedly stealth
mega-cap-tech vehicles that inflate my true Tech exposure. On MY
basket, real data says the opposite: Tech goes from ~50% (naive,
lumping "Tech ETF" with equity Tech) **down** to ~44% (x-ray).

Why? Because QQQ is only ~50% Tech by weight — the rest is
Communication Services (GOOG, META), Consumer Cyclical (AMZN, TSLA),
Healthcare, and so on. Calling QQQ a "Tech ETF" over-counts Tech;
unwrapping it redistributes to those other sectors.

That's actually the more interesting lesson: **naive sector labels
overstate concentration too**. If you were about to trim what you
thought was Tech overexposure and replace it with an "ETF for
diversification," the look-through says you may not have had the
Tech overexposure you thought.

*The code cell below re-renders the sector view post-look-through,
alongside the naive view + delta.*

In [5]:
# [Phase B / NB03 §4] Sector view WITH look-through — the pivot
# Same sector classification as §2, but on the flattened portfolio.
# The story: raw Tech ~38% becomes something notably higher after
# the ETF names are unwrapped.
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

# Cache profile lookups — the same underlying names appear across ETFs
_profile_cache: dict[str, str] = {}
def _sector_for(sym: str) -> str:
    if sym in NAIVE_SECTOR_LABELS:
        return NAIVE_SECTOR_LABELS[sym]
    if sym.startswith("BOND_"):
        return "Bond Fund"
    if sym in _profile_cache:
        return _profile_cache[sym]
    try:
        info = obb.equity.profile(symbol=sym, provider="fmp_cached").to_df()
        sector = info["sector"].iloc[0] if "sector" in info.columns else "Unknown"
    except Exception:
        sector = "Unknown"
    _profile_cache[sym] = sector
    return sector

# Flatten to sector weights AFTER look-through
xray_by_sector: dict[str, float] = {}
for sym, w in effective.items():
    sector = _sector_for(sym)
    xray_by_sector[sector] = xray_by_sector.get(sector, 0.0) + w

print("Look-through sector view (ETFs flattened to underlying names):")
print(f"{'Sector':<24}{'Weight':>10}")
print("-" * 36)
for sector, w in sorted(xray_by_sector.items(), key=lambda kv: -kv[1]):
    print(f"{sector:<24}{w*100:>9.2f}%")

print()
print("Side-by-side delta (naive vs look-through):")
print(f"{'Sector':<24}{'Naive':>10}{'X-Ray':>10}{'Delta':>10}")
print("-" * 56)
all_sectors = set(naive_by_sector) | set(xray_by_sector)
for sec in sorted(all_sectors, key=lambda s: -xray_by_sector.get(s, 0)):
    n = naive_by_sector.get(sec, 0.0) * 100
    x = xray_by_sector.get(sec, 0.0) * 100
    delta = x - n
    print(f"{sec:<24}{n:>9.1f}%{x:>9.2f}%{delta:>+9.1f}%")

tech_xray = xray_by_sector.get("Technology", 0.0) + xray_by_sector.get("Tech ETF", 0.0)
print()
print(f"Effective Tech exposure: {tech_xray*100:.1f}%  "
      f"(was {tech_naive*100:.1f}% naive — that\'s the hidden concentration)")


Look-through sector view (ETFs flattened to underlying names):
Sector                      Weight
------------------------------------
Technology                  43.80%
Bond Fund                   10.00%
Communication Services       9.79%
Real Estate                  3.16%
Commodity                    3.00%
Consumer Cyclical            1.74%
Unknown                      1.15%

Side-by-side delta (naive vs look-through):
Sector                       Naive     X-Ray     Delta
--------------------------------------------------------
Technology                   36.0%    43.80%     +7.8%
Bond Fund                    10.0%    10.00%     +0.0%
Communication Services        8.0%     9.79%     +1.8%
Real Estate                   0.0%     3.16%     +3.2%
Commodity                     3.0%     3.00%     +0.0%
Consumer Cyclical             0.0%     1.74%     +1.7%
Unknown                       0.0%     1.15%     +1.1%
Broad ETF                    20.0%     0.00%    -20.0%
REIT                   

## 5. HHI + effective-N — two numbers to quote from now on

Two concentration metrics worth learning:

- **HHI (Herfindahl-Hirschman Index)** — sum of squared weights.
  Ranges from 1/N (perfectly diversified across N names) to 1 (all in
  one name). Higher = more concentrated.
- **Effective N** — `1 / HHI`. Answers "how many *equivalent*
  equal-weight positions am I holding?" Not a standard Investopedia term; it's the intuitive reciprocal of HHI that every portfolio-management text uses (Grinold & Kahn call it the "effective breadth" of the book).

Prediction from the story I'd heard: HHI goes UP post-look-through
because unwrapping ETFs reveals hidden concentration. On MY basket,
real data says **the opposite** — HHI drops from ~0.12 raw to ~0.07
x-ray, and Effective-N rises from ~8 to ~14. Look-through
DILUTES my concentration here because QQQ etc spread across many
small positions, so what looked like 3 big ETF bets is actually 40+
tiny slivers of well-known names.

The two numbers replace "I hold 10 things" as the sentence I answer
with when someone asks "how concentrated is your book?" The answer
now: "I hold 10 tickers but under look-through my effective count is
~14 equal-weight positions, with HHI ~0.07 (moderate, not
concentrated)."

> **📖 Herfindahl-Hirschman Index (HHI)** — originally an antitrust measure of market concentration; sum of squared market shares. Portfolio managers borrow it directly, substituting position weights for market shares. The DOJ calls an industry "unconcentrated" below 1500 (0.15 in decimal); the same threshold reads well for equity books. [Investopedia →](https://www.investopedia.com/terms/h/hhi.asp)

*The code cell below computes HHI and effective-N for both the raw
and the flattened views.*

In [6]:
# [Phase B / NB03 §5] HHI + effective-N — the two numbers to quote from now on
# HHI = sum of squared weights. Effective-N = 1/HHI.
# For raw view: use the basket rows as-is.
# For x-ray view: use the effective-position weights.

def _hhi(weights) -> float:
    return sum(w * w for w in weights)

# Raw
raw_weights = [p["weight"] for p in basket]
hhi_raw = _hhi(raw_weights)
neff_raw = 1.0 / hhi_raw if hhi_raw > 0 else float("nan")

# X-ray
xray_weights = list(effective.values())
hhi_xray = _hhi(xray_weights)
neff_xray = 1.0 / hhi_xray if hhi_xray > 0 else float("nan")

print(f"{'Metric':<24}{'Raw':>12}{'X-Ray':>12}{'Delta':>12}")
print("-" * 60)
print(f"{'HHI':<24}{hhi_raw:>12.4f}{hhi_xray:>12.4f}{hhi_xray-hhi_raw:>+12.4f}")
print(f"{'Effective-N':<24}{neff_raw:>12.2f}{neff_xray:>12.2f}{neff_xray-neff_raw:>+12.2f}")
print()
print(f"Interpretation:")
print(f"  I hold {len(basket)} tickers. Effective-N under look-through is {neff_xray:.1f} —")
print(f"  meaning my concentration is equivalent to holding {neff_xray:.1f} equal-weight positions,")
print(f"  not the {len(basket)} I thought I did.")


Metric                           Raw       X-Ray       Delta
------------------------------------------------------------
HHI                           0.1206      0.0693     -0.0513
Effective-N                     8.29       14.44       +6.15

Interpretation:
  I hold 10 tickers. Effective-N under look-through is 14.4 —
  meaning my concentration is equivalent to holding 14.4 equal-weight positions,
  not the 10 I thought I did.


## 6. Risk metrics — `obb.portfolio_intel.risk.metrics`

The concept primer for this section: risk numbers are trader
seatbelts. You don't need them when nothing goes wrong; you need
them when everything goes wrong at once. **Sharpe** answers *am I
being paid for the ride I'm on* (return per unit of pain). **Vol**
answers *how bumpy is the ride* (annualized standard deviation of
daily returns). **Max drawdown** answers *what's the worst hole I've
been in*, which is the number your gut actually cares about at 3am.
**Tracking error** answers *am I actually managing a portfolio or
just tilting SPY* — if my tracking error is 1% my book is basically
the index with extra costs. Rules of thumb: Sharpe above ~1 is decent
retail, above ~2 is suspicious, above ~3 is a bug. Annualized vol
below ~12% is "sleep well," 12-20% is "equity book," above ~25% is
"you have a concentration or leverage problem." Max drawdown of
-20% is what the S&P 500 does every few years; -50%+ is 2008 or
the dot-com bust. Tracking error of 4-8% is what an active manager
who's actually doing something looks like. The platform enforces
these with a **hardened refusal path** (§8): if the covariance
matrix is rank-deficient because 3 of 10 names are missing prices,
you get a refusal, not a silent zero.

Four numbers you should never operate without:

- **Sharpe ratio** — annualized excess return per unit of vol. Above 1
  is decent, above 2 is suspicious, above 3 is either a genius or a
  bug.
- **Annualized volatility** — the ± you should expect on your book.
- **Maximum drawdown** — the worst peak-to-trough decline over the
  lookback. Ask yourself: could I actually sit through this?
- **Tracking error vs SPY** — how far your book wanders from the
  benchmark. High tracking error is only worth it if your Sharpe
  clears the benchmark's.

> **📖 Sharpe ratio** — `(portfolio return − risk-free rate) / portfolio volatility`, annualized. Named for Bill Sharpe (Nobel 1990); it's the reference risk-adjusted return. [Investopedia →](https://www.investopedia.com/terms/s/sharperatio.asp)
>
> **📖 Volatility** — the annualized standard deviation of returns, i.e. how much the book bounces around its mean. Not the same thing as *risk of loss* — a straight-line 30%-a-year winner has vol too. [Investopedia →](https://www.investopedia.com/terms/v/volatility.asp)
>
> **📖 Maximum drawdown** — largest peak-to-trough loss over the lookback window, expressed as a percentage of the prior peak. The "how much did this hurt at its worst?" number. Recovery time (how long to make it back) is the second half of that story. [Investopedia →](https://www.investopedia.com/terms/m/maximum-drawdown-mdd.asp)
>
> **📖 Tracking error** — the standard deviation of the *difference* between portfolio return and benchmark return. Passive index funds sit near 0; active managers who deviate meaningfully live in the 4-8% range. [Investopedia →](https://www.investopedia.com/terms/t/trackingerror.asp)
>
> **📖 Benchmark** — the index you compare to. Wrong benchmark = misleading numbers: a small-cap growth book measured against SPY looks brilliant in a small-cap year and terrible in a mega-cap year for reasons unrelated to skill. SPY is the default here because the basket is US-equity-heavy. [Investopedia →](https://www.investopedia.com/terms/b/benchmark.asp)
>
> **📖 Risk-free rate** — the return you could have earned with no risk (T-bill yield is the standard proxy). Sharpe subtracts this because excess return above cash is the only return worth pricing. [Investopedia →](https://www.investopedia.com/terms/r/risk-freerate.asp)

*The code cell below computes all four on the basket and renders them
against SPY for context.*

In [7]:
# [Phase B / NB03 §6] Risk metrics — Sharpe / vol / MaxDD / tracking-error vs SPY
# Uses obb.portfolio_intel.risk.metrics. Requires:
#   - basket: list[{symbol, weight}]
#   - returns_source: dict[symbol -> list[float]]  (per-position daily returns)
#   - benchmark_returns: list[float] (daily returns of the benchmark)
#
# We build returns from EquityHistorical via fmp_cached — this is the
# fmp_cached-primary tier working end-to-end.
import time
import numpy as np
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

WINDOW_DAYS = 60  # ~3 months of trading days — enough for stable stats, fast to fetch

# Fetch history for each basket symbol + benchmark
symbols_to_fetch = [p["symbol"] for p in basket] + ["SPY"]

def _daily_returns(sym: str) -> list[float]:
    df = obb.equity.price.historical(
        symbol=sym,
        provider="fmp_cached",
        start_date="2026-05-01",
        end_date="2026-07-24",
    ).to_df()
    if df.empty or "close" not in df.columns:
        return []
    closes = df["close"].dropna().tolist()
    if len(closes) < 2:
        return []
    return list(np.diff(closes) / closes[:-1])

t0 = time.perf_counter()
returns_source = {}
missing = []
for sym in symbols_to_fetch:
    r = _daily_returns(sym)
    if r:
        returns_source[sym] = r
    else:
        missing.append(sym)
print(f"Fetched returns for {len(returns_source)}/{len(symbols_to_fetch)} symbols "
      f"in {time.perf_counter()-t0:.1f}s")
if missing:
    print(f"  (missing: {missing} — these will be dropped from risk calc)")

benchmark_returns = returns_source.pop("SPY", [])

# Trim series to the same length + drop any position without returns
common_len = min(len(v) for v in returns_source.values()) if returns_source else 0
if common_len == 0:
    print("Cannot compute risk metrics — no returns available.")
else:
    returns_source = {k: v[:common_len] for k, v in returns_source.items()}
    benchmark_returns = benchmark_returns[:common_len]
    # Restrict basket to positions with returns
    basket_for_risk = [p for p in basket if p["symbol"] in returns_source]
    # Renormalize weights
    w_total = sum(p["weight"] for p in basket_for_risk)
    basket_for_risk = [
        {"symbol": p["symbol"], "weight": p["weight"] / w_total}
        for p in basket_for_risk
    ]
    print(f"Computing risk on {len(basket_for_risk)} positions × {common_len} daily returns")

    r = obb.portfolio_intel.risk.metrics(
        basket=basket_for_risk,
        returns_source=returns_source,
        benchmark_returns=benchmark_returns,
    )
    res = r.results
    print()
    print("Portfolio-level risk (parametric, ~3 months of daily returns):")
    for name in sorted(dir(res)):
        if name.startswith("_") or callable(getattr(res, name, None)):
            continue
        val = getattr(res, name)
        if isinstance(val, (int, float)):
            print(f"  {name+':':<24}{val:>12.4f}")
        else:
            print(f"  {name+':':<24}{val}")


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Failed to fetch dividends for NVDA: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Failed to fetch dividends for GOOGL: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Failed to fetch dividends for AAPL: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Failed to fetch dividends for AMD: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


...

Failed to fetch dividends for VNQ: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Failed to fetch dividends for BND: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Failed to fetch dividends for GLD: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Failed to fetch dividends for SPY: Invalid variable type: value should be str, int or float, got datetime.date(2026, 5, 1) of type <class 'datetime.date'>


Fetched returns for 11/11 symbols in 0.5s
Computing risk on 10 positions × 57 daily returns

Portfolio-level risk (parametric, ~3 months of daily returns):
  beta:                         1.1965
  cvar_95:                      0.0223
  model_computed_fields:  {}
  model_config:           {}
  model_extra:            None
  model_fields:           {'volatility': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, description='Portfolio std-dev (per-period, same time-scale as input returns). None when the basket has any symbol missing from returns_source.'), 'var_95': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, description='Parametric VaR at 95% confidence (loss magnitude, positive). Computed as z * volatility.'), 'cvar_95': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, description='Parametric CVaR at 95% (expected shortfall, positive). Computed as phi(z) / (1 - Phi(z)) * volatility.'), 'beta': FieldInfo(a

## 7. Concentration — `obb.portfolio_intel.risk.concentration`

Rounds out the picture with:

- **Top-K weights** (K = 1, 3, 5) — how much of the book is in the
  top handful of effective positions
- **Single-name kill-shot** — what happens to portfolio value if the
  single largest effective position drops 20%

For a basket that *looks* like 10 positions but is effectively
5-6 mega-cap tech names, the top-1 weight is going to surprise you.

> **📖 Correlation** — how tightly two return series move together, on a scale of -1 to +1. Two 0.9-correlated positions are barely two positions in a risk sense; that's why the effective-N in §5 is more honest than the ticker count. [Investopedia →](https://www.investopedia.com/terms/c/correlation.asp)

*The code cell below runs `.concentration` and prints the top-K table
plus the kill-shot number.*

In [8]:
# [Phase B / NB03 §7] Concentration — top-K weights + single-name kill-shot
# NOTE: obb.portfolio_intel.risk.concentration internally calls
# xray.look_through, which stalls without the ETF-Holdings sub-plan
# (same issue as §3). We compute the concentration metrics
# in-notebook on the effective positions from §3 — same math the
# router does, just against the snapshot-derived flatten.

def _topk_weight(effective_dict: dict, k: int) -> float:
    """Sum of the top-k weights."""
    return sum(sorted(effective_dict.values(), reverse=True)[:k])

top1 = _topk_weight(effective, 1)
top5 = _topk_weight(effective, 5)
top10 = _topk_weight(effective, 10)

print("Concentration (post-look-through, computed in-notebook):")
print(f"  {'HHI':<24}{hhi_xray:>10.4f}")
print(f"  {'Effective-N':<24}{neff_xray:>10.2f}")
print(f"  {'Top-1 weight':<24}{top1*100:>9.2f}%")
print(f"  {'Top-5 weight':<24}{top5*100:>9.2f}%")
print(f"  {'Top-10 weight':<24}{top10*100:>9.2f}%")

# Single-name kill-shot on the largest effective position
top_sym, top_w = max(effective.items(), key=lambda kv: kv[1])
kill_shot_pct = top_w * 0.20  # 20% drop scenario on largest effective position
print()
print(f"Story-side single-name kill-shot:")
print(f"  Largest effective position: {top_sym} at {top_w*100:.1f}%")
print(f"  If {top_sym} drops 20%: portfolio hit = {kill_shot_pct*100:.2f}%")
print()
print("(obb.portfolio_intel.risk.concentration router IS available and does")
print(" the same math — but it re-calls xray.look_through internally, which")
print(" stalls on the ETF-Holdings-exhausted path for me. Computing directly")
print(" from the effective_positions we already have is faster and honest.)")


Concentration (post-look-through, computed in-notebook):
  HHI                         0.0693
  Effective-N                  14.44
  Top-1 weight                14.01%
  Top-5 weight                55.94%
  Top-10 weight               68.17%

Story-side single-name kill-shot:
  Largest effective position: MSFT at 14.0%
  If MSFT drops 20%: portfolio hit = 2.80%

(obb.portfolio_intel.risk.concentration router IS available and does
 the same math — but it re-calls xray.look_through internally, which
 stalls on the ETF-Holdings-exhausted path for me. Computing directly
 from the effective_positions we already have is faster and honest.)


## 8. Silent-failure guards — why the numbers can be trusted

`obb.portfolio_intel.risk` was hardened in PR #905 against three ways
these metrics silently return garbage:

1. **Partial-book variance** — if only 7 of 10 positions have valid
   price history, the covariance matrix is undefined. Old code
   substituted a rank-deficient matrix; new code refuses and tells
   you which positions failed.
2. **NaN in covariance / returns / benchmark** — same-shape refusal.
3. **NaN / Inf prices in the input** — refused at the door.

*The code cell below deliberately corrupts one price series in the
basket and re-runs `.metrics` to show the refusal path fires (not a
silent-zero result).*

In [9]:
# [Phase B / NB03 §8] Silent-failure guard — NaN in prices must refuse, not silent-zero
# From PR #905 hardening: portfolio_intel.risk.metrics refuses to
# silently compute against NaN-poisoned inputs. Demonstrate by
# poisoning one position's returns with NaN and asserting the guard fires.
import math
import copy
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

# Clone the returns_source we built above; poison the first position
if returns_source and len(returns_source) > 1:
    poisoned = copy.deepcopy(returns_source)
    victim = list(poisoned.keys())[0]
    poisoned[victim] = [math.nan] * len(poisoned[victim])
    print(f"Poisoning {victim}'s return series with all-NaN values...")
    try:
        r = obb.portfolio_intel.risk.metrics(
            basket=basket_for_risk,
            returns_source=poisoned,
            benchmark_returns=benchmark_returns,
        )
        # If it didn't raise, check whether the metrics look right
        res = r.results
        vol = getattr(res, "annualized_volatility", None)
        if vol is None or (isinstance(vol, float) and math.isnan(vol)):
            print(f"  OK: guard fired — vol reported as {vol} (loud-empty)")
        else:
            print(f"  UNEXPECTED: computation succeeded with vol={vol} — check silent-guard behavior")
    except Exception as exc:
        print(f"  OK: guard raised {type(exc).__name__}: {str(exc)[:120]}")
else:
    print("(skipped — need returns_source populated from §6)")


Poisoning MSFT's return series with all-NaN values...
  OK: guard fired — vol reported as None (loud-empty)


## 9. Reading the numbers

A quick reference — the two-sentence version of each metric, in Sam's
words:

- **HHI**: sum of squared weights. Higher means more concentrated. Ours
  roughly doubles when we look through the ETFs.
- **Effective N**: 1/HHI. "How many equal-weight positions am I really
  holding?"
- **Sharpe**: excess return over risk-free per unit of vol. Under
  ~0.5, your risk isn't being paid for.
- **Max drawdown**: worst historical peak-to-trough. If it hurts to
  read the number, it will hurt more to live through it.
- **Tracking error**: how far you wander from your benchmark. Only worth
  it if your Sharpe beats the benchmark's.

## 10. Save state for NB04, NB05, NB06

Everything downstream picks up:
- `.notebook_state/basket.json` — the 10 positions (already written)
- `.notebook_state/xray.pkl` — the flattened positions + sector weights
- `.notebook_state/risk.pkl` — the four risk metrics

*The code cell below pickles the x-ray + risk artifacts.*

In [10]:
# [Phase B / NB03 §9] Save state for NB04/NB05/NB06
# Pickle safety note (same as NB02 §9): file lives under
# .notebook_state/ (gitignored), trusted-local-only, never shipped.
# Long-term: msgspec/pydantic schema would remove pickle entirely.
import pickle  # noqa: S403  # trusted local artifact; safety documented above
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)

# xray artifact — dict form (simpler to consume in later notebooks)
xray_artifact = {
    "basket": basket,
    "effective_positions": effective,  # dict[str,float]  post-look-through
    "naive_by_sector": naive_by_sector,
    "xray_by_sector": xray_by_sector,
    "hhi_raw": hhi_raw,
    "hhi_xray": hhi_xray,
    "neff_raw": neff_raw,
    "neff_xray": neff_xray,
}
(state / "xray.pkl").write_bytes(pickle.dumps(xray_artifact))

# risk artifact — only saved if the risk cell succeeded
try:
    risk_artifact = {
        "basket_for_risk": basket_for_risk,
        "returns_source": returns_source,
        "benchmark_returns": benchmark_returns,
        # r.results from §6 — attribute-carrying object
        "metrics_dump": {
            name: getattr(r.results, name)
            for name in dir(r.results)
            if not name.startswith("_") and not callable(getattr(r.results, name, None))
        },
    }
    (state / "risk.pkl").write_bytes(pickle.dumps(risk_artifact))
    saved_risk = True
except NameError:
    saved_risk = False

print(f"Wrote (repo-rel):  {(state/'xray.pkl')}    {(state/'xray.pkl').stat().st_size:,} bytes")
if saved_risk:
    print(f"Wrote (repo-rel):  {(state/'risk.pkl')}    {(state/'risk.pkl').stat().st_size:,} bytes")
else:
    print("(risk.pkl not written — §6 didn't complete)")


Wrote (repo-rel):  .notebook_state\xray.pkl    1,447 bytes
Wrote (repo-rel):  .notebook_state\risk.pkl    15,828 bytes


---

## What is NOT in this notebook

- **Cash positions.** Sam holds equity + ETFs only in this basket; cash-as-a-position support is [#903](https://github.com/prajoria/OpenBB/issues/903) and not shipped.
- **Short positions.** Same story — [#904](https://github.com/prajoria/OpenBB/issues/904).
- **Factor decomposition (Fama-French, Carhart).** The `openbb_famafrench` extension exists but isn't wired into portfolio_intel risk yet.

## Preview of NB04

Now I know what I own and how concentrated I am. The picture is worse than I thought. But the picture is static — it's a snapshot of my exposures. Two things move that snapshot every week: **events** on the calendar (earnings, dividends, splits) and **smart-money activity** (13F changes, insider transactions, government trades). In NB04 we overlay both.


## 📚 Further reading

Every Investopedia link cited in this notebook, plus canonical references:

- [Portfolio — Investopedia](https://www.investopedia.com/terms/p/portfolio.asp)
- [Position — Investopedia](https://www.investopedia.com/terms/p/position.asp)
- [Weighted average — Investopedia](https://www.investopedia.com/terms/w/weighted.asp)
- [Diversification — Investopedia](https://www.investopedia.com/terms/d/diversification.asp)
- [Sector breakdown — Investopedia](https://www.investopedia.com/terms/s/sectorbreakdown.asp)
- [Look-through — Investopedia](https://www.investopedia.com/terms/l/look-through-earnings.asp)
- [Herfindahl-Hirschman Index — Investopedia](https://www.investopedia.com/terms/h/hhi.asp)
- [Sharpe ratio — Investopedia](https://www.investopedia.com/terms/s/sharperatio.asp)
- [Volatility — Investopedia](https://www.investopedia.com/terms/v/volatility.asp)
- [Maximum drawdown — Investopedia](https://www.investopedia.com/terms/m/maximum-drawdown-mdd.asp)
- [Tracking error — Investopedia](https://www.investopedia.com/terms/t/trackingerror.asp)
- [Benchmark — Investopedia](https://www.investopedia.com/terms/b/benchmark.asp)
- [Risk-free rate — Investopedia](https://www.investopedia.com/terms/r/risk-freerate.asp)
- [Correlation — Investopedia](https://www.investopedia.com/terms/c/correlation.asp)

**Canonical references beyond Investopedia:**

- Grinold, R. C. & Kahn, R. N. — *Active Portfolio Management*, 2nd ed.,
  ch. 3 ("Expected Returns and the Arithmetic of Active Management"). The
  reference text for the effective-breadth / effective-N intuition and the
  Information Ratio framing that generalizes Sharpe for active books.
- Markowitz, H. — "Portfolio Selection," *Journal of Finance* 7(1), 1952.
  The original mean-variance paper; every risk number in this notebook
  descends from it.


## 📚 Further reading

Every Investopedia link cited in this notebook, plus canonical references:

- [Portfolio — Investopedia](https://www.investopedia.com/terms/p/portfolio.asp)
- [Position — Investopedia](https://www.investopedia.com/terms/p/position.asp)
- [Weighted average — Investopedia](https://www.investopedia.com/terms/w/weighted.asp)
- [Diversification — Investopedia](https://www.investopedia.com/terms/d/diversification.asp)
- [Sector breakdown — Investopedia](https://www.investopedia.com/terms/s/sectorbreakdown.asp)
- [Look-through — Investopedia](https://www.investopedia.com/terms/l/look-through-earnings.asp)
- [Herfindahl-Hirschman Index — Investopedia](https://www.investopedia.com/terms/h/hhi.asp)
- [Sharpe ratio — Investopedia](https://www.investopedia.com/terms/s/sharperatio.asp)
- [Volatility — Investopedia](https://www.investopedia.com/terms/v/volatility.asp)
- [Maximum drawdown — Investopedia](https://www.investopedia.com/terms/m/maximum-drawdown-mdd.asp)
- [Tracking error — Investopedia](https://www.investopedia.com/terms/t/trackingerror.asp)
- [Benchmark — Investopedia](https://www.investopedia.com/terms/b/benchmark.asp)
- [Risk-free rate — Investopedia](https://www.investopedia.com/terms/r/risk-freerate.asp)
- [Correlation — Investopedia](https://www.investopedia.com/terms/c/correlation.asp)

**Canonical references beyond Investopedia:**

- Grinold, R. C. & Kahn, R. N. — *Active Portfolio Management*, 2nd ed.,
  ch. 3 ("Expected Returns and the Arithmetic of Active Management"). The
  reference text for the effective-breadth / effective-N intuition and the
  Information Ratio framing that generalizes Sharpe for active books.
- Markowitz, H. — "Portfolio Selection," *Journal of Finance* 7(1), 1952.
  The original mean-variance paper; every risk number in this notebook
  descends from it.
